# Kahaani: SDXL + IP-Adapter pages for the consistency experiment

Runs on a free **Kaggle** (or Colab) **T4 GPU**. For each eval story it draws every page with
SDXL base + IP-Adapter, conditioned on the story's **shared hero image** (the same image
the other setups are scored against) and using the **same page prompts** as the app
(`kahaani.pipeline.illustrator.page_prompt`) and the same per-book seed (`book_seed`).
So the only thing that changes versus the `fixed` setup is the model and the reference image.

Output: `eval/out/images/ipadapter/{id}/p{n}.png`, zipped for download. Copy the folder back into
your local `eval/out/images/` and run `python -m eval.consistency score-clip` / `score-judge`.

**Settings on Kaggle:** Accelerator = GPU T4, Internet = on.

## 1. Install libraries
Kaggle already has torch. `kahaani` itself only needs pydantic for the prompt helpers.

In [ ]:
%pip install -q "diffusers>=0.31" "transformers>=4.44" accelerate safetensors pydantic python-dotenv

## 2. Get the code and the inputs

The notebook needs three things: the `kahaani` package (for `page_prompt`), `eval/out/stories.jsonl`,
and the hero images in `eval/out/images/hero/{id}.png`. `eval/out/` is git-ignored, so the
stories and heroes always come from a dataset upload.

**Option A: clone the repo** (set `REPO_URL`), then add the stories + heroes as a dataset.

**Option B: upload everything as a Kaggle dataset** named `kahaani-eval` containing
`kahaani/` (the package folder), `stories.jsonl` and `hero/` (the PNGs). Add it via
*Add Input → Your Datasets*; it appears under `/kaggle/input/kahaani-eval/`.

Make the dataset zip locally with:
```bash
zip -r kahaani-eval.zip kahaani eval/out/stories.jsonl eval/out/images/hero
```

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/<you>/kahaani.git"; leave empty for Option B
DATASET = Path("/kaggle/input/kahaani-eval")  # Colab: wherever you unzipped the upload
WORK = Path("/kaggle/working/kahaani") if Path("/kaggle").exists() else Path("kahaani_work")

if REPO_URL and not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
WORK.mkdir(parents=True, exist_ok=True)


# Copy whatever the dataset has into the same layout as the repo.
def find(name):
    hits = list(DATASET.rglob(name)) if DATASET.exists() else []
    return hits[0] if hits else None


if not (WORK / "kahaani").exists() and (pkg := find("illustrator.py")):
    shutil.copytree(pkg.parents[1], WORK / "kahaani")
out = WORK / "eval" / "out"
(out / "images" / "hero").mkdir(parents=True, exist_ok=True)
if not (out / "stories.jsonl").exists() and (f := find("stories.jsonl")):
    shutil.copy(f, out / "stories.jsonl")
if (hero_dir := find("hero")) and hero_dir.is_dir():
    for png in hero_dir.glob("*.png"):
        shutil.copy(png, out / "images" / "hero" / png.name)

sys.path.insert(0, str(WORK))
os.chdir(WORK)
print(
    "stories:",
    (out / "stories.jsonl").exists(),
    "| heroes:",
    len(list((out / "images/hero").glob("*.png"))),
)

## 3. Load the stories and the app's prompt helpers
We only use stories that have a hero image, so every setup is scored on the same books.

In [ ]:
import json

from kahaani.pipeline.illustrator import book_seed, page_prompt
from kahaani.schemas import Brief, Story

IMAGES = Path("eval/out/images")
stories = []
for line in Path("eval/out/stories.jsonl").read_text("utf-8").splitlines():
    row = json.loads(line)
    if row.get("status") == "ok" and (IMAGES / "hero" / f"{row['id']}.png").exists():
        stories.append(
            (row["id"], Brief.model_validate(row["brief"]), Story.model_validate(row["story"]))
        )
print(len(stories), "stories with a hero image")

MAX_PAGES = 4  # match `generate --max-pages` used for the other setups
LIMIT = None  # set e.g. 5 for a quick run

## 4. Load SDXL + IP-Adapter for a 16 GB T4

* fp16 weights and the fp16-safe VAE (the stock SDXL VAE can produce black images in fp16).
* `ip-adapter_sdxl.bin` from `h94/IP-Adapter` (folder `sdxl_models`), which feeds the hero picture
  through a CLIP image encoder into extra cross-attention layers.
* `enable_model_cpu_offload()` keeps only the active sub-model on the GPU, and
  `enable_attention_slicing()` trades a little speed for lower peak memory. Together they keep SDXL +
  IP-Adapter well inside a T4's 16 GB (about 30-60 s per 1024px image).

In [ ]:
import torch
from diffusers import AutoencoderKL, StableDiffusionXLPipeline

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", vae=vae, torch_dtype=torch.float16, variant="fp16"
)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
print("loaded")

## 5. Draw the pages
Same negative prompt as `kahaani/providers/image_diffusers.py`. Existing files are skipped, so you
can re-run after a session timeout.

In [ ]:
import time

from PIL import Image

NEGATIVE = "text, letters, words, watermark, signature, scary, blurry"


def draw(brief, story, page, hero, scale, steps=30, size=1024):
    pipe.set_ip_adapter_scale(scale)
    gen = torch.Generator("cpu").manual_seed(book_seed(story))
    return pipe(
        prompt=page_prompt(brief, story, page),
        negative_prompt=NEGATIVE,
        ip_adapter_image=hero,
        num_inference_steps=steps,
        width=size,
        height=size,
        generator=gen,
    ).images[0]


def run(setup, scale, items):
    for sid, brief, story in items:
        hero = Image.open(IMAGES / "hero" / f"{sid}.png").convert("RGB")
        folder = IMAGES / setup / sid
        folder.mkdir(parents=True, exist_ok=True)
        for page in story.pages[:MAX_PAGES]:
            path = folder / f"p{page.number}.png"
            if path.exists():
                continue
            t = time.perf_counter()
            draw(brief, story, page, hero, scale).save(path)
            print(f"{setup} {sid} p{page.number}: {time.perf_counter() - t:.0f}s", flush=True)


run("ipadapter", 0.6, stories[:LIMIT])

## 6. Scale sweep: 0.4 / 0.6 / 0.8 on 3 stories

`ip_adapter_scale` sets how much the reference image steers the result versus the text prompt:

* **≈0.4**: the prompt dominates. Scenes, poses and the companion follow the story well, but the child
  drifts (hair, outfit colours).
* **≈0.6**: the usual middle ground, and the main setting above.
* **≈0.8**: strong likeness, but the model starts copying the hero picture itself (standing pose, plain
  background) and ignores parts of the scene.

The sweep folders (`ipadapter_s0.4`, ...) are scored as extra setups by `eval.consistency`, so the
trade-off shows up in the CLIP and judge numbers too.

In [ ]:
SWEEP = [0.4, 0.6, 0.8]
for scale in SWEEP:
    run(f"ipadapter_s{scale}", scale, stories[:3])

In [ ]:
import matplotlib.pyplot as plt

sid = stories[0][0]
fig, axes = plt.subplots(len(SWEEP), MAX_PAGES + 1, figsize=(3 * (MAX_PAGES + 1), 3 * len(SWEEP)))
for r, scale in enumerate(SWEEP):
    axes[r][0].imshow(Image.open(IMAGES / "hero" / f"{sid}.png"))
    axes[r][0].set_title("hero" if r == 0 else "")
    axes[r][0].set_ylabel(f"scale {scale}")
    for c in range(1, MAX_PAGES + 1):
        p = IMAGES / f"ipadapter_s{scale}" / sid / f"p{c}.png"
        if p.exists():
            axes[r][c].imshow(Image.open(p))
    for ax in axes[r]:
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()

## 7. Zip for download
Download `ipadapter_images.zip` from the Output panel, unzip it into your local `eval/out/images/`.

In [ ]:
import zipfile

zip_path = (
    Path("/kaggle/working/ipadapter_images.zip")
    if Path("/kaggle").exists()
    else Path("ipadapter_images.zip")
)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in IMAGES.glob("ipadapter*"):
        for f in folder.rglob("*.png"):
            z.write(f, f.relative_to(IMAGES))
print(zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")